# 01 — Dataproc Standard Spark Medallion Benchmark (`spark_standard`)

Executes the 3-stage TPC-DS Medallion pipeline (`Stage 1: Bronze -> Silver`, `Stage 2A: Vectorized Tiering`, `Stage 2B: Scalar Python UDF Contrast`, `Stage 3: 5-Way Star Join + Rolling 7d/30d Gold Mart`) on a **Dataproc on Compute Engine (`2.3-debian12`, x86)** cluster configured with **`--engine=default`**.

### Benchmark Fairness & Parity Guarantees (`SPEC.md`)
- **Zero BigQuery Spark connector pushdown:** Reads Bronze Parquet and writes Silver/Gold Parquet strictly over Cloud Storage (`gs://${BENCH_BUCKET}/...`).
- **Identical PySpark pipeline:** Executes `scripts/run_spark_medallion.py` (`--engine-variant spark_standard`), sharing 100% identical DataFrame transformations with `02_medallion_spark_lightning.ipynb`.
- **Warm-up before timing:** Excludes YARN executor startup via `spark.range(1).count()` prior to Stage 1 timing; captures output row counts, `checksum_net_paid`, and Catalyst formatted plans outside stage timers.


## 1. Load `.env` & Resolve Benchmark Configuration

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
import json
import os
from pathlib import Path
import re
import subprocess
import sys
import time

from dotenv import load_dotenv
from google.cloud import storage
from IPython.display import display
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "SPEC.md").exists() else cwd.parent
if not (REPO_ROOT / "SPEC.md").exists():
    raise FileNotFoundError(
        f"Could not locate SPEC.md from {cwd}. Run notebook from repository root or notebooks/."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.metrics_helper import infer_vcpus_from_machine_type, log_stage_metric

dotenv_path = REPO_ROOT / ".env"
if dotenv_path.exists():
    load_dotenv(dotenv_path=dotenv_path, override=False)


def _strip_mtls_prefix(raw: str) -> str:
    lines = [
        line for line in raw.splitlines()
        if not line.startswith("Created enterprise-certificate-proxy configuration file")
    ]
    return "\n".join(lines).strip()


def _infer_worker_vcpus(machine_type: str, num_workers: int, *, allow_override: bool = True) -> int:
    if allow_override:
        override = os.environ.get("TOTAL_WORKER_VCPUS", "").strip()
        if override:
            return int(override)
    return infer_vcpus_from_machine_type(machine_type) * int(num_workers)


PROJECT_ID = os.environ.get("PROJECT_ID", "").strip()
if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "PROJECT_ID is unset or placeholder. Copy .env.example to .env and configure PROJECT_ID."
    )

PROJECT_NUMBER = os.environ.get("PROJECT_NUMBER", "").strip()
if not PROJECT_NUMBER:
    proj_raw = subprocess.check_output(
        ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
        text=True,
    )
    PROJECT_NUMBER = _strip_mtls_prefix(proj_raw)
if not PROJECT_NUMBER:
    raise ValueError("Unable to resolve PROJECT_NUMBER.")

BENCH_BUCKET = os.environ.get("BENCH_BUCKET", "").strip().removeprefix("gs://").strip("/")
if not BENCH_BUCKET:
    BENCH_BUCKET = f"lakehouse-bench-{PROJECT_NUMBER}"

DATAPROC_REGION = os.environ.get("DATAPROC_REGION", "us-central1").strip() or "us-central1"
CLUSTER_STD_NAME = os.environ.get("CLUSTER_STD_NAME", "bench-spark-std-x86").strip() or "bench-spark-std-x86"
SCALE_PRESET = os.environ.get("SCALE_PRESET", "small_1g").strip() or "small_1g"
if SCALE_PRESET not in ("small_1g", "medium_100g", "large_500g"):
    raise ValueError(f"Unsupported SCALE_PRESET={SCALE_PRESET!r}")

WORKER_MACHINE_TYPE = os.environ.get("WORKER_MACHINE_TYPE", "n2d-standard-8").strip() or "n2d-standard-8"
NUM_PRIMARY_WORKERS = int(os.environ.get("NUM_PRIMARY_WORKERS", "2"))
TOTAL_WORKER_VCPUS = _infer_worker_vcpus(WORKER_MACHINE_TYPE, NUM_PRIMARY_WORKERS)

RUN_PYTHON_UDF_STAGE = (
    "true"
    if os.environ.get("RUN_PYTHON_UDF_STAGE", "true").strip().lower() in ("1", "true", "yes")
    else "false"
)
UDF_MAX_SAMPLE_ROWS = int(os.environ.get("UDF_MAX_SAMPLE_ROWS", "500000"))

DS_METRICS = os.environ.get("DS_METRICS", "bench_metrics").strip() or "bench_metrics"
WRITE_TO_BQ_METRICS = os.environ.get("WRITE_TO_BQ_METRICS", "true").strip().lower() in (
    "1",
    "true",
    "yes",
)

ENGINE_VARIANT = "spark_standard"
STORAGE_BACKEND = "gcs_parquet"
RUN_ID = (
    os.environ.get("BENCH_RUN_ID", "").strip()
    or os.environ.get("RUN_ID", "").strip()
    or f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{SCALE_PRESET}"
)
METRICS_GCS_URI = f"gs://{BENCH_BUCKET}/metrics/{RUN_ID}/{ENGINE_VARIANT}.json"
LIVE_JSONL_PATH = REPO_ROOT / "results" / "live_runs" / "benchmark_runs.jsonl"
RUNNER_SCRIPT = REPO_ROOT / "scripts" / "run_spark_medallion.py"

config_summary = pd.DataFrame(
    [
        ("RUN_ID", RUN_ID),
        ("ENGINE_VARIANT", ENGINE_VARIANT),
        ("SCALE_PRESET", SCALE_PRESET),
        ("DATAPROC_REGION", DATAPROC_REGION),
        ("CLUSTER_STD_NAME", CLUSTER_STD_NAME),
        ("WORKER_MACHINE_TYPE", WORKER_MACHINE_TYPE),
        ("NUM_PRIMARY_WORKERS", str(NUM_PRIMARY_WORKERS)),
        ("TOTAL_WORKER_VCPUS", str(TOTAL_WORKER_VCPUS)),
        ("RUN_PYTHON_UDF_STAGE", RUN_PYTHON_UDF_STAGE),
        ("UDF_MAX_SAMPLE_ROWS", str(UDF_MAX_SAMPLE_ROWS)),
        ("BENCH_BUCKET", f"gs://{BENCH_BUCKET}"),
        ("METRICS_GCS_URI", METRICS_GCS_URI),
        ("WRITE_TO_BQ_METRICS", str(WRITE_TO_BQ_METRICS)),
    ],
    columns=["Parameter", "Value"],
)
display(config_summary)


,Parameter,Value
0,RUN_ID,20261116T103813Z_large_500g
1,ENGINE_VARIANT,spark_standard
2,SCALE_PRESET,large_500g
3,DATAPROC_REGION,us-central1
4,CLUSTER_STD_NAME,bench-spark-std-x86
5,WORKER_MACHINE_TYPE,n2d-standard-16
6,NUM_PRIMARY_WORKERS,4
7,TOTAL_WORKER_VCPUS,64
8,RUN_PYTHON_UDF_STAGE,true
9,UDF_MAX_SAMPLE_ROWS,500000


## 2. Verify Dataproc Standard Cluster Engine (`--engine=default`) & Hardware Parity

In [2]:
EXECUTION_MODE = os.environ.get("SPARK_EXECUTION_MODE", "").strip().lower()
if not EXECUTION_MODE:
    try:
        from pyspark.sql import SparkSession

        EXECUTION_MODE = (
            "in_cluster" if SparkSession.getActiveSession() is not None else "dataproc_submit"
        )
    except ImportError:
        EXECUTION_MODE = "dataproc_submit"

if EXECUTION_MODE not in ("dataproc_submit", "in_cluster"):
    raise ValueError(f"Unsupported EXECUTION_MODE={EXECUTION_MODE!r}")

print(f"Execution mode: {EXECUTION_MODE}")

if EXECUTION_MODE == "dataproc_submit":
    raw_desc = subprocess.check_output(
        [
            "gcloud",
            "dataproc",
            "clusters",
            "describe",
            CLUSTER_STD_NAME,
            f"--region={DATAPROC_REGION}",
            f"--project={PROJECT_ID}",
            "--format=json",
        ],
        text=True,
    )
    json_start = raw_desc.find("{")
    if json_start < 0:
        raise RuntimeError(f"Unexpected gcloud describe output: {raw_desc[:300]}")
    cluster_desc = json.loads(raw_desc[json_start:])

    cluster_state = cluster_desc.get("status", {}).get("state", "UNKNOWN")
    if cluster_state != "RUNNING":
        raise RuntimeError(
            f"Cluster {CLUSTER_STD_NAME} is in state {cluster_state!r}; expected 'RUNNING'."
        )

    cfg = cluster_desc.get("config", {})
    soft_cfg = cfg.get("softwareConfig", {})
    soft_props = soft_cfg.get("properties", {})
    cluster_engine = cfg.get("engine") or soft_props.get("dataproc:dataproc.engine", "DEFAULT")

    if "LIGHTNING" in str(cluster_engine).upper():
        raise RuntimeError(
            f"Cluster {CLUSTER_STD_NAME} reports engine={cluster_engine!r}; expected Standard ('DEFAULT')."
        )
    if soft_props.get("spark:spark.dataproc.lightningEngine.runtime") == "native":
        raise RuntimeError(
            f"Cluster {CLUSTER_STD_NAME} unexpectedly enables Lightning NQE runtime=native."
        )

    master_cfg = cfg.get("masterConfig", {})
    worker_cfg = cfg.get("workerConfig", {})
    actual_worker_type = str(worker_cfg.get("machineTypeUri", "")).rsplit("/", 1)[-1]
    actual_workers = int(worker_cfg.get("numInstances", 0))
    actual_ssds = int(worker_cfg.get("diskConfig", {}).get("numLocalSsds", 0))

    if not actual_worker_type or actual_workers <= 0:
        raise RuntimeError(
            f"Unable to derive worker shape from Dataproc cluster {CLUSTER_STD_NAME}: "
            f"machineType={actual_worker_type!r}, numInstances={actual_workers}."
        )
    if actual_worker_type.startswith(("t2a-", "c4a-")):
        raise RuntimeError(
            f"ARM worker machine type {actual_worker_type!r} violates x86 parity (SPEC.md §1.3)."
        )

    live_worker_vcpus = _infer_worker_vcpus(
        actual_worker_type, actual_workers, allow_override=False
    )
    assert actual_worker_type == WORKER_MACHINE_TYPE, (
        f"Cluster worker type {actual_worker_type!r} does not match .env WORKER_MACHINE_TYPE={WORKER_MACHINE_TYPE!r}."
    )
    assert actual_workers == NUM_PRIMARY_WORKERS, (
        f"Cluster worker count {actual_workers} does not match .env NUM_PRIMARY_WORKERS={NUM_PRIMARY_WORKERS}."
    )
    assert live_worker_vcpus == TOTAL_WORKER_VCPUS, (
        f"Derived live worker vCPUs ({live_worker_vcpus}) != .env TOTAL_WORKER_VCPUS ({TOTAL_WORKER_VCPUS})."
    )
    TOTAL_WORKER_VCPUS = live_worker_vcpus
    WORKER_MACHINE_TYPE = actual_worker_type

    cluster_hw_df = pd.DataFrame(
        [
            ("Cluster State", cluster_state),
            ("Dataproc Engine", str(cluster_engine)),
            ("Image Version", str(soft_cfg.get("imageVersion", ""))),
            ("Master Machine Type", str(master_cfg.get("machineTypeUri", "")).rsplit("/", 1)[-1]),
            ("Worker Machine Type", WORKER_MACHINE_TYPE),
            ("Primary Worker Count", str(actual_workers)),
            ("Live Total Worker vCPUs", str(TOTAL_WORKER_VCPUS)),
            ("Worker Local SSDs", str(actual_ssds)),
        ],
        columns=["Cluster Property", "Observed Value"],
    )
    display(cluster_hw_df)
else:
    from pyspark.sql import SparkSession

    active_spark = SparkSession.builder.getOrCreate()
    runtime_prop = active_spark.conf.get("spark.dataproc.lightningEngine.runtime", "unset")
    if runtime_prop == "native":
        raise RuntimeError("Expected Standard Spark kernel, but lightningEngine.runtime=native is active.")
    print(
        f"Attached to active SparkSession (app={active_spark.sparkContext.appName}, lightningEngine.runtime={runtime_prop})."
    )


Execution mode: dataproc_submit


,Cluster Property,Observed Value
0,Cluster State,RUNNING
1,Dataproc Engine,DEFAULT
2,Image Version,2.3.38-debian12
3,Master Machine Type,n2d-standard-4
4,Worker Machine Type,n2d-standard-16
5,Primary Worker Count,4
6,Live Total Worker vCPUs,64
7,Worker Local SSDs,2


## 3. Pre-Flight Check: Open Bronze Parquet in GCS & Zero BigQuery Connector Audit

In [3]:
if not RUNNER_SCRIPT.exists():
    raise FileNotFoundError(f"Missing shared runner: {RUNNER_SCRIPT}")

runner_source = RUNNER_SCRIPT.read_text(encoding="utf-8")
bq_connector_pattern = re.compile(
    r"\.format\(\s*['\"]big" + r"query['\"]\s*\)|spark-big" + r"query"
)
if bq_connector_pattern.search(runner_source):
    raise AssertionError(f"Fairness violation: BigQuery Spark connector reference found in {RUNNER_SCRIPT}")
print(f"Fairness check passed: {RUNNER_SCRIPT.name} uses pure GCS Parquet I/O (zero BigQuery connector).")

BRONZE_TABLES = (
    "store_sales",
    "store_returns",
    "web_sales",
    "web_returns",
    "customer",
    "customer_demographics",
    "item",
    "date_dim",
)

gcs_client = storage.Client(project=PROJECT_ID)
bucket = gcs_client.bucket(BENCH_BUCKET)
bronze_inventory = []
missing_tables = []

for table in BRONZE_TABLES:
    prefix = f"bronze/{SCALE_PRESET}/{table}/"
    parquet_blobs = [
        b for b in gcs_client.list_blobs(bucket, prefix=prefix)
        if not b.name.endswith("/") and b.name.endswith(".parquet")
    ]
    if not parquet_blobs:
        missing_tables.append(prefix)
    total_bytes = sum(int(b.size or 0) for b in parquet_blobs)
    bronze_inventory.append(
        {
            "table": table,
            "gcs_prefix": f"gs://{BENCH_BUCKET}/{prefix}",
            "parquet_files": len(parquet_blobs),
            "size_mib": round(total_bytes / (1024 ** 2), 2),
        }
    )

bronze_df = pd.DataFrame(bronze_inventory)
display(bronze_df)

if missing_tables:
    raise FileNotFoundError(
        "Missing Bronze Parquet data for tables: "
        + ", ".join(missing_tables)
        + ". Run notebooks/00_data_copy_ingestion.ipynb first."
    )


Fairness check passed: run_spark_medallion.py uses pure GCS Parquet I/O (zero BigQuery connector).


,table,gcs_prefix,parquet_files,size_mib
0,store_sales,gs://lakehouse-bench-/bronze/large...,1758,54824.80
1,store_returns,gs://lakehouse-bench-/bronze/large...,468,10703.69
2,web_sales,gs://lakehouse-bench-/bronze/large...,1289,27069.99
3,web_returns,gs://lakehouse-bench-/bronze/large...,104,3075.10
4,customer,gs://lakehouse-bench-/bronze/large...,166,3462.68
5,customer_demographics,gs://lakehouse-bench-/bronze/large...,5,12.93
6,item,gs://lakehouse-bench-/bronze/large...,1,44.03
7,date_dim,gs://lakehouse-bench-/bronze/large...,1,2.46


## 4. Submit Shared PySpark Medallion Job (`scripts/run_spark_medallion.py`)

In [4]:
captured_stdout_lines: list[str] = []
in_memory_payload = None
submit_start_epoch = time.time()

if EXECUTION_MODE == "dataproc_submit":
    submit_cmd = [
        "gcloud",
        "dataproc",
        "jobs",
        "submit",
        "pyspark",
        str(RUNNER_SCRIPT),
        f"--cluster={CLUSTER_STD_NAME}",
        f"--region={DATAPROC_REGION}",
        f"--project={PROJECT_ID}",
        "--",
        "--engine-variant",
        ENGINE_VARIANT,
        "--scale-preset",
        SCALE_PRESET,
        "--bench-bucket",
        BENCH_BUCKET,
        "--run-id",
        RUN_ID,
        "--worker-machine-type",
        WORKER_MACHINE_TYPE,
        "--total-worker-vcpus",
        str(TOTAL_WORKER_VCPUS),
        "--run-python-udf",
        RUN_PYTHON_UDF_STAGE,
        "--udf-max-sample-rows",
        str(UDF_MAX_SAMPLE_ROWS),
        "--metrics-gcs-uri",
        METRICS_GCS_URI,
    ]
    print("Submitting Dataproc PySpark job:")
    print(" ", " ".join(submit_cmd))

    proc = subprocess.Popen(
        submit_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        captured_stdout_lines.append(line.rstrip("\n"))
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Dataproc PySpark job failed with exit code {rc}.")
else:
    from pyspark.sql import SparkSession
    from scripts.run_spark_medallion import run_pipeline, write_json_to_hadoop_uri

    spark = SparkSession.builder.getOrCreate()
    in_memory_payload = run_pipeline(
        spark=spark,
        engine_variant=ENGINE_VARIANT,
        scale_preset=SCALE_PRESET,
        bronze_base_uri=f"gs://{BENCH_BUCKET}/bronze/{SCALE_PRESET}",
        silver_base_uri=f"gs://{BENCH_BUCKET}/silver/{SCALE_PRESET}/{ENGINE_VARIANT}",
        gold_base_uri=f"gs://{BENCH_BUCKET}/gold/{SCALE_PRESET}/{ENGINE_VARIANT}",
        run_id=RUN_ID,
        worker_machine_type=WORKER_MACHINE_TYPE,
        total_worker_vcpus=TOTAL_WORKER_VCPUS,
        run_python_udf=(RUN_PYTHON_UDF_STAGE == "true"),
        udf_max_sample_rows=UDF_MAX_SAMPLE_ROWS,
    )
    write_json_to_hadoop_uri(spark, METRICS_GCS_URI, in_memory_payload)
    print(f"In-cluster Medallion execution completed and wrote {METRICS_GCS_URI}")

submit_wall_clock_sec = round(time.time() - submit_start_epoch, 4)
print(f"End-to-end submission wall clock (submit_wall_clock_sec): {submit_wall_clock_sec:.2f}s")


Submitting Dataproc PySpark job:
  gcloud dataproc jobs submit pyspark ./scripts/run_spark_medallion.py --cluster=bench-spark-std-x86 --region=us-central1 --project=<your-gcp-project-id> -- --engine-variant spark_standard --scale-preset large_500g --bench-bucket lakehouse-bench-<project-number> --run-id 20261116T103813Z_large_500g --worker-machine-type n2d-standard-16 --total-worker-vcpus 64 --run-python-udf true --udf-max-sample-rows 500000 --metrics-gcs-uri gs://lakehouse-bench-<project-number>/metrics/20261116T103813Z_large_500g/spark_standard.json


Job [7d760140a4e0452397a7df8d4aec8324] submitted.
Waiting for job output...


26/11/16 10:47:52 INFO SparkEnv: Registering MapOutputTracker
26/11/16 10:47:52 INFO SparkEnv: Registering BlockManagerMaster
26/11/16 10:47:52 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/11/16 10:47:52 INFO SparkEnv: Registering OutputCommitCoordinator
26/11/16 10:47:53 INFO MetricsConfig: Loaded properties from hadoop-metrics2.properties
26/11/16 10:47:53 INFO MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
26/11/16 10:47:53 INFO MetricsSystemImpl: google-hadoop-file-system metrics system started


26/11/16 10:47:53 INFO DataprocSparkPlugin: Registered 202 driver metrics
26/11/16 10:47:53 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at bench-spark-std-x86-m.us-central1-a.c.<your-gcp-project-id>.internal./10.128.0.109:8032
26/11/16 10:47:53 INFO AHSProxy: Connecting to Application History server at bench-spark-std-x86-m.us-central1-a.c.<your-gcp-project-id>.internal./10.128.0.109:10200
26/11/16 10:47:54 INFO Configuration: resource-types.xml not found
26/11/16 10:47:54 INFO ResourceUtils: Unable to find 'resource-types.xml'.
26/11/16 10:47:54 INFO YarnClientImpl: Submitted application application_1794823906440_0002


26/11/16 10:47:55 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at bench-spark-std-x86-m.us-central1-a.c.<your-gcp-project-id>.internal./10.128.0.109:8030
26/11/16 10:47:57 INFO GhfsGlobalStorageStatistics: periodic connector metrics: {gcs_api_client_non_found_response_count=1, gcs_api_client_side_error_count=1, gcs_api_time=251, gcs_api_total_request_count=2, gcs_connector_time=355, gcs_list_file_request=1, gcs_list_file_request_duration=128, gcs_list_file_request_max=128, gcs_list_file_request_mean=128, gcs_list_file_request_min=128, gcs_metadata_request=1, gcs_metadata_request_duration=123, gcs_metadata_request_max=123, gcs_metadata_request_mean=123, gcs_metadata_request_min=123, gs_filesystem_create=3, gs_filesystem_initialize=2, op_get_file_status=1, op_get_file_status_duration=355, op_get_file_status_max=355, op_get_file_status_mean=355, op_get_file_status_min=355, uptimeSeconds=4} [CONTEXT ratelimit_period="5 MINUTES" ]
26/11/16 10:47:57 INFO GoogleCloud

26/11/16 10:47:58 INFO GoogleHadoopOutputStream: hflush(): No-op due to rate limit (RateLimiter[stableRate=0.2qps]): readers will *not* yet see flushed data for gs://dataproc-temp-us-central1-<project-number>-k2ftoqe7/cd977287-8ccb-4b1d-af8b-64b969cd13a2/spark-job-history/application_1794823906440_0002.inprogress [CONTEXT ratelimit_period="1 MINUTES" ]
26/11/16 10:47:58 INFO ArgumentParser: Couldn't log config from file, will read it from SparkConf
26/11/16 10:47:58 INFO ArgumentParser: Couldn't log config from file, will read it from SparkConf
26/11/16 10:47:59 INFO OpenLineageClientUtils: Disabled facets: [debug, spark.logicalPlan, spark_unknown]


... (24 outputs left out)

26/11/16 10:55:12 INFO GoogleCloudStorageFileSystemImpl: Successfully repaired 'gs://lakehouse-bench-<project-number>/gold/large_500g/spark_standard/gold_omnichannel_kpi_mart/' directory.
26/11/16 10:55:13 INFO EventEmitter: Emitting lineage completed successfully with run id: 01a1e3fd-9393-77a3-b891-c8181073f432
26/11/16 10:55:13 INFO RddExecutionContext: Config field is not HadoopMapRedWriteConfigUtil or HadoopMapReduceWriteConfigUtil, it's org.apache.spark.rdd.RDD$$Lambda$4391/0x0000000801d6d040
26/11/16 10:55:13 INFO RddExecutionContext: Found job conf from RDD Configuration: core-default.xml, core-site.xml, mapred-default.xml, mapred-site.xml, yarn-default.xml, yarn-site.xml, resource-types.xml, hdfs-default.xml, hdfs-rbf-default.xml, hdfs-site.xml, hdfs-rbf-site.xml
26/11/16 10:55:13 INFO RddExecutionContext: Found output path null from RDD MapPartitionsRDD[138] at parquet at NativeMethodAccessorImpl.java:0
26/11/16 10:55:13 INFO EventEmitter: Emitting lineage completed successfu

26/11/16 10:55:13 INFO SparkSQLExecutionContext: OpenLineage received Spark event that is configured to be skipped: SparkListenerJobEnd
26/11/16 10:55:13 INFO SparkSQLExecutionContext: OpenLineage received Spark event that is configured to be skipped: SparkListenerJobStart
26/11/16 10:55:13 INFO SparkSQLExecutionContext: OpenLineage received Spark event that is configured to be skipped: SparkListenerSQLExecutionEnd
BENCHMARK_METRICS_JSON_BEGIN
{
  "run_id": "20261116T103813Z_large_500g",
  "engine_variant": "spark_standard",
  "scale_preset": "large_500g",
  "lightning_runtime_conf": "default",
  "udf_max_sample_rows": 500000,
  "records": [
    {
      "run_id": "20261116T103813Z_large_500g",
      "timestamp_utc": "2026-11-16T10:51:11Z",
      "scale_preset": "large_500g",
      "engine_variant": "spark_standard",
      "storage_backend": "gcs_parquet",
      "stage_id": "stage1_bronze_to_silver",
      "start_epoch_sec": 1794826084.850061,
      "end_epoch_sec": 1794826263.40559,
  

Job [7d760140a4e0452397a7df8d4aec8324] finished successfully.
done: true
driverControlFilesUri: gs://lakehouse-bench-<project-number>/google-cloud-dataproc-metainfo/cd977287-8ccb-4b1d-af8b-64b969cd13a2/jobs/7d760140a4e0452397a7df8d4aec8324/
driverOutputResourceUri: gs://lakehouse-bench-<project-number>/google-cloud-dataproc-metainfo/cd977287-8ccb-4b1d-af8b-64b969cd13a2/jobs/7d760140a4e0452397a7df8d4aec8324/driveroutput
jobUuid: 47a40b1b-944c-300f-bd23-4c6137a1cc52
placement:
  clusterName: bench-spark-std-x86
  clusterUuid: cd977287-8ccb-4b1d-af8b-64b969cd13a2
pysparkJob:
  args:
  - --engine-variant
  - spark_standard
  - --scale-preset
  - large_500g
  - --bench-bucket
  - lakehouse-bench-<project-number>
  - --run-id
  - 20261116T103813Z_large_500g
  - --worker-machine-type
  - n2d-standard-16
  - --total-worker-vcpus
  - '64'
  - --run-python-udf
  - 'true'
  - --udf-max-sample-rows
  - '500000'
  - --metrics-gcs-uri
  - gs://lakehouse-bench-<project-number>/metrics/20261116T103813

End-to-end submission wall clock (submit_wall_clock_sec): 460.36s


## 5. Retrieve GCS Telemetry JSON & Inspect Catalyst Physical Plans (`WholeStageCodegen`)

In [5]:
def _extract_payload_from_stdout(lines: list[str]) -> dict | None:
    try:
        begin_idx = lines.index("BENCHMARK_METRICS_JSON_BEGIN")
        end_idx = lines.index("BENCHMARK_METRICS_JSON_END")
    except ValueError:
        return None
    return json.loads("\n".join(lines[begin_idx + 1 : end_idx]))


if in_memory_payload is not None:
    metrics_payload = in_memory_payload
else:
    gcs_blob_path = METRICS_GCS_URI.removeprefix(f"gs://{BENCH_BUCKET}/")
    blob = bucket.blob(gcs_blob_path)
    if blob.exists():
        metrics_payload = json.loads(blob.download_as_text(encoding="utf-8"))
        print(f"Loaded metrics JSON from {METRICS_GCS_URI}")
    else:
        metrics_payload = _extract_payload_from_stdout(captured_stdout_lines)
        if metrics_payload is None:
            raise FileNotFoundError(
                f"Neither {METRICS_GCS_URI} nor stdout JSON delimiters were found."
            )
        print("Loaded metrics JSON from Dataproc job stdout fallback.")

print(
    f"Observed spark.dataproc.lightningEngine.runtime = {metrics_payload.get('lightning_runtime_conf', 'unset')!r}"
)

OPERATORS_TO_COUNT = (
    "WholeStageCodegen",
    "Scan parquet",
    "Filter",
    "Project",
    "HashAggregate",
    "SortMergeJoin",
    "BroadcastHashJoin",
    "Exchange",
    "Window",
    "BatchEvalPythonExec",
)

plans = metrics_payload.get("plans", {})
operator_rows = []
for stage_id, plan_text in plans.items():
    row = {"stage_id": stage_id}
    for op in OPERATORS_TO_COUNT:
        row[op] = plan_text.count(op)
    operator_rows.append(row)

print("\nCatalyst Physical Plan Operator Summary (Standard Spark JVM):")
display(pd.DataFrame(operator_rows))

for stage_id, plan_text in plans.items():
    print(f"\n{'=' * 80}\nSTAGE: {stage_id}\n{'=' * 80}")
    print(plan_text)


Loaded metrics JSON from gs://lakehouse-bench-<project-number>/metrics/20261116T103813Z_large_500g/spark_standard.json
Observed spark.dataproc.lightningEngine.runtime = 'default'

Catalyst Physical Plan Operator Summary (Standard Spark JVM):


,stage_id,WholeStageCodegen,Scan parquet,Filter,Project,HashAggregate,SortMergeJoin,BroadcastHashJoin,Exchange,Window,BatchEvalPythonExec
0,stage1_bronze_to_silver,0,8,16,12,12,6,0,12,0,0
1,stage2a_silver_vectorized,0,2,0,2,0,0,0,0,0,0
2,stage2b_silver_udf,0,2,3,2,0,0,0,0,0,0
3,stage3_silver_to_gold,0,10,20,18,12,3,9,24,6,0



STAGE: stage1_bronze_to_silver
== Physical Plan ==
AdaptiveSparkPlan (65)
+- == Final Plan ==
   Execute InsertIntoHadoopFsRelationCommand (39)
   +- WriteFiles (38)
      +- Union (37)
         :- * Project (17)
         :  +- * SortMergeJoin LeftOuter (16)
         :     :- * Sort (6)
         :     :  +- ShuffleQueryStage (5), Statistics(sizeInBytes=65.4 GiB, rowCount=7.98E+8)
         :     :     +- Exchange (4)
         :     :        +- * Filter (3)
         :     :           +- * ColumnarToRow (2)
         :     :              +- Scan parquet  (1)
         :     +- * Sort (15)
         :        +- * HashAggregate (14)
         :           +- ShuffleQueryStage (13), Statistics(sizeInBytes=8.4 GiB, rowCount=1.87E+8)
         :              +- Exchange (12)
         :                 +- * HashAggregate (11)
         :                    +- * Project (10)
         :                       +- * Filter (9)
         :                          +- * ColumnarToRow (8)
         :          

## 6. Persist Telemetry (`metrics_helper.py`) & Verify Silver/Gold Parquet Outputs

In [6]:
logged_records = []
for rec in metrics_payload["records"]:
    logged = log_stage_metric(
        run_id=rec["run_id"],
        scale_preset=rec["scale_preset"],
        engine_variant=rec["engine_variant"],
        storage_backend=rec["storage_backend"],
        stage_id=rec["stage_id"],
        start_epoch_sec=float(rec["start_epoch_sec"]),
        end_epoch_sec=float(rec["end_epoch_sec"]),
        output_row_count=int(rec["output_row_count"]),
        checksum_net_paid=rec.get("checksum_net_paid"),
        worker_machine_type=rec.get("worker_machine_type"),
        total_worker_vcpus=rec.get("total_worker_vcpus"),
        bq_slot_cap=None,
        bq_total_slot_ms=None,
        bq_bytes_processed=None,
        nqe_enabled=bool(rec.get("nqe_enabled", False)),
        nqe_fallback_note=str(rec.get("nqe_fallback_note", "")),
        est_list_cost_usd=rec.get("est_list_cost_usd"),
        sample_fraction=float(rec.get("sample_fraction", 1.0)),
        jsonl_path=LIVE_JSONL_PATH,
        write_to_bq=WRITE_TO_BQ_METRICS,
        project_id=PROJECT_ID,
        ds_metrics=DS_METRICS,
    )
    logged_records.append(logged)

summary_cols = [
    "stage_id",
    "wall_clock_sec",
    "vcpu_seconds",
    "output_row_count",
    "checksum_net_paid",
    "sample_fraction",
    "est_list_cost_usd",
]
summary_df = pd.DataFrame(logged_records)[summary_cols]
print(f"Logged {len(logged_records)} stage records via scripts.metrics_helper to {LIVE_JSONL_PATH}")
display(summary_df)

in_cluster_stage_sum_sec = round(sum(float(r["wall_clock_sec"]) for r in logged_records), 4)
submit_overhead_sec = round(max(0.0, submit_wall_clock_sec - in_cluster_stage_sum_sec), 4)
timing_boundary_df = pd.DataFrame(
    [
        ("Warm In-Cluster Stage Sum (wall_clock_sec)", in_cluster_stage_sum_sec),
        ("End-to-End Submit Wall Clock (submit_wall_clock_sec)", submit_wall_clock_sec),
        ("Submit / YARN Startup / Post-Timer Audit Overhead (sec)", submit_overhead_sec),
    ],
    columns=["Timing Boundary Metric", "Seconds"],
)
print(
    "\nTimer Boundary Breakdown (stage wall_clock_sec excludes gcloud submit, YARN ramp, and post-stage checksum scans):"
)
display(timing_boundary_df)

output_prefixes = [
    ("stage1_bronze_to_silver", f"silver/{SCALE_PRESET}/{ENGINE_VARIANT}/silver_omnichannel_sales/"),
    ("stage2a_silver_vectorized", f"silver/{SCALE_PRESET}/{ENGINE_VARIANT}/silver_enriched_sales/"),
    ("stage3_silver_to_gold", f"gold/{SCALE_PRESET}/{ENGINE_VARIANT}/gold_omnichannel_kpi_mart/"),
]
if RUN_PYTHON_UDF_STAGE == "true":
    output_prefixes.insert(
        2,
        ("stage2b_silver_udf", f"silver/{SCALE_PRESET}/{ENGINE_VARIANT}/silver_udf_benchmark_out/"),
    )

output_audit = []
for stage_id, prefix in output_prefixes:
    blobs = [
        b for b in gcs_client.list_blobs(bucket, prefix=prefix)
        if not b.name.endswith("/") and b.name.endswith(".parquet")
    ]
    total_bytes = sum(int(b.size or 0) for b in blobs)
    output_audit.append(
        {
            "stage_id": stage_id,
            "gcs_uri": f"gs://{BENCH_BUCKET}/{prefix}",
            "parquet_files": len(blobs),
            "size_mib": round(total_bytes / (1024 ** 2), 2),
        }
    )

print("\nMaterialized Silver/Gold Parquet Outputs:")
display(pd.DataFrame(output_audit))


Logged 4 stage records via scripts.metrics_helper to ./results/live_runs/benchmark_runs.jsonl


,stage_id,wall_clock_sec,vcpu_seconds,output_row_count,checksum_net_paid,sample_fraction,est_list_cost_usd
0,stage1_bronze_to_silver,178.5555,11427.5520,1014392792,1.841730e+12,1.000000,0.190905
1,stage2a_silver_vectorized,97.3811,6232.3904,1014392792,1.841730e+12,1.000000,0.104116
2,stage2b_silver_udf,31.6357,2024.6848,50660213,9.200404e+10,0.049941,0.033824
3,stage3_silver_to_gold,106.1238,6791.9232,28068,1.837184e+12,1.000000,0.113464



Timer Boundary Breakdown (stage wall_clock_sec excludes gcloud submit, YARN ramp, and post-stage checksum scans):


,Timing Boundary Metric,Seconds
0,Warm In-Cluster Stage Sum (wall_clock_sec),413.6961
1,End-to-End Submit Wall Clock (submit_wall_cloc...,460.3628
2,Submit / YARN Startup / Post-Timer Audit Overh...,46.6667



Materialized Silver/Gold Parquet Outputs:


,stage_id,gcs_uri,parquet_files,size_mib
0,stage1_bronze_to_silver,gs://lakehouse-bench-/silver/large...,1200,47066.93
1,stage2a_silver_vectorized,gs://lakehouse-bench-/silver/large...,434,52533.68
2,stage2b_silver_udf,gs://lakehouse-bench-/silver/large...,434,2718.61
3,stage3_silver_to_gold,gs://lakehouse-bench-/gold/large_5...,2,2.10
